Decision trees and ensemble methods - Aiden Ward, Grady Algire, Brady Cooper

Fashion-MNIST (classification) and California Housing (regression). Runs top to bottom with fixed seeds, takes about 6 hours. Figures go to `figs/`, csvs go to `results/`, writeup is in the report.

In [1]:
# setup
import os, time, gzip, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import (BaggingClassifier, BaggingRegressor, RandomForestClassifier, RandomForestRegressor,
                              ExtraTreesClassifier, ExtraTreesRegressor, AdaBoostClassifier, AdaBoostRegressor,
                              HistGradientBoostingClassifier, GradientBoostingRegressor,
                              VotingClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (accuracy_score, f1_score, mean_squared_error, mean_absolute_error,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
from sklearn.datasets import fetch_california_housing, make_moons

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
os.makedirs('figs', exist_ok=True)
os.makedirs('results', exist_ok=True)

N_SEEDS = 5  # 5 seeds for the single tree stuff and the final table (big sweeps only use 3)

plt.rcParams['figure.dpi'] = 90

# saves to figs/ so we can use them in the report
def savefig(name):
    plt.tight_layout()
    plt.savefig(f'figs/{name}.png', dpi=130, bbox_inches='tight')
    plt.show()

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

def fmt(m, s, d=2):
    return f'{m:.{d}f} ± {s:.{d}f}'


In [2]:
# load fashion mnist
# fetch_openml('Fashion-MNIST') kept timing out (openml was down) so this grabs the
# original files from the fashion-mnist github instead. same data, same train/test split
CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

def load_fashion():
    base = 'https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/'
    os.makedirs('data/fashion', exist_ok=True)
    out = []
    for name in ['train-images-idx3-ubyte', 'train-labels-idx1-ubyte',
                 't10k-images-idx3-ubyte', 't10k-labels-idx1-ubyte']:
        path = f'data/fashion/{name}.gz'
        if not os.path.exists(path):
            print('downloading', name)
            urllib.request.urlretrieve(base + name + '.gz', path)
        with gzip.open(path, 'rb') as f:
            raw = f.read()
        if 'images' in name:
            out.append(np.frombuffer(raw, np.uint8, offset=16).reshape(-1, 784))
        else:
            out.append(np.frombuffer(raw, np.uint8, offset=8).astype(int))
    return out

X_all, y_all, X_test, y_test = load_fashion()
X_all = X_all.astype(np.float32) / 255
X_test = X_test.astype(np.float32) / 255

# fixed stratified subsample: 10k train, 5k val. test set isnt touched until part 6
X_tr, X_rest, y_tr, y_rest = train_test_split(X_all, y_all, train_size=10000, stratify=y_all, random_state=SEED)
X_val, _, y_val, _ = train_test_split(X_rest, y_rest, train_size=5000, stratify=y_rest, random_state=SEED)

print('train', X_tr.shape, 'val', X_val.shape, 'test', X_test.shape)
print('train class counts', np.bincount(y_tr))

# quick look at the data
fig, axes = plt.subplots(2, 10, figsize=(12, 3))
for k in range(10):
    idx = np.where(y_tr == k)[0][:2]
    for r in range(2):
        axes[r, k].imshow(X_tr[idx[r]].reshape(28, 28), cmap='gray_r')
        axes[r, k].axis('off')
    axes[0, k].set_title(CLASS_NAMES[k], fontsize=8)
plt.suptitle('Fashion-MNIST examples (2 per class)')
savefig('data_examples')

train (10000, 784) val (5000, 784) test (10000, 784)
train class counts [1000 1000 1000 1000 1000 1000 1000 1000 1000 1000]


In [3]:
# load california housing (60/20/20 split)
cal = fetch_california_housing(data_home='data')
Xc, yc = cal.data, cal.target
feat_c = list(cal.feature_names)
Xc_trval, Xc_test, yc_trval, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=SEED)
Xc_tr, Xc_val, yc_tr, yc_val = train_test_split(Xc_trval, yc_trval, test_size=0.25, random_state=SEED)
print('california train', Xc_tr.shape, 'val', Xc_val.shape, 'test', Xc_test.shape)
print(feat_c)

california train (12384, 8) val (4128, 8) test (4128, 8)
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


In [4]:
# helper functions
def subsample_idx(n, seed, frac=0.8):
    # single trees are basically deterministic, so to get error bars
    # each seed trains on a different random 80% of the training set
    rng = np.random.RandomState(seed)
    return rng.choice(n, int(frac * n), replace=False)

def count_nodes(model):
    # total number of tree nodes in a model (tree, bagging, forest, boosting)
    if hasattr(model, 'tree_'):
        return model.tree_.node_count
    if hasattr(model, '_predictors'):  # hist gradient boosting stores its trees here
        return sum(len(p.nodes) for it in model._predictors for p in it)
    ests = model.estimators_
    if isinstance(ests, np.ndarray):  # GradientBoostingRegressor has a 2d array
        ests = ests.ravel()
    return sum(count_nodes(e) for e in ests)

In [5]:
# part 1a - by hand
def gini(counts):
    p = np.array(counts) / np.sum(counts)
    return 1 - np.sum(p ** 2)

def entropy(counts):
    p = np.array(counts) / np.sum(counts)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

def misclass(counts):
    p = np.array(counts) / np.sum(counts)
    return 1 - p.max()

def impurity_decrease(parent, children, f):
    n = np.sum(parent)
    return f(parent) - sum(np.sum(c) / n * f(c) for c in children)

parent, left, right = [25, 15], [20, 2], [5, 13]
for name, f in [('gini', gini), ('entropy', entropy), ('misclass', misclass)]:
    print(f'{name:9s} parent={f(parent):.4f} left={f(left):.4f} right={f(right):.4f} '
          f'decrease={impurity_decrease(parent, [left, right], f):.4f}')

gini      parent=0.4688 left=0.1653 right=0.4012 decrease=0.1973
entropy   parent=0.9544 left=0.4395 right=0.8524 decrease=0.3291
misclass  parent=0.3750 left=0.0909 right=0.2778 decrease=0.2000


In [6]:
# part 1a - impurity plot
p = np.linspace(0, 1, 501)
G = 2 * p * (1 - p)
H = -(p * np.log2(np.clip(p, 1e-12, 1)) + (1 - p) * np.log2(np.clip(1 - p, 1e-12, 1)))
E = np.minimum(p, 1 - p)
plt.figure(figsize=(5, 3.5))
plt.plot(p, G, label='Gini  G')
plt.plot(p, H / 2, label='Entropy / 2  (H/2)')
plt.plot(p, E, label='Misclassification error')
plt.xlabel('p (fraction of class 1)')
plt.ylabel('impurity')
plt.title('Two-class impurity measures')
plt.legend()
savefig('p1a_impurity')

# the classic example where misclassification can't tell two splits apart
par = [400, 400]
splitA = [[300, 100], [100, 300]]
splitB = [[200, 400], [200, 0]]
for name, f in [('gini', gini), ('entropy', entropy), ('misclass', misclass)]:
    print(f'{name:9s} split A: {impurity_decrease(par, splitA, f):.4f}   split B: {impurity_decrease(par, splitB, f):.4f}')

gini      split A: 0.1250   split B: 0.1667
entropy   split A: 0.1887   split B: 0.3113
misclass  split A: 0.2500   split B: 0.2500


In [7]:
# part 1b - depth sweep
DEPTHS = [2, 4, 6, 8, 10, 12, 15, 20, None]
DEPTH_LABELS = [str(d) for d in DEPTHS]

rows = []
full_trees = {}   # unlimited depth trees, used to compare which features get picked
for crit in ['gini', 'entropy']:
    for d in DEPTHS:
        for s in range(N_SEEDS):
            idx = subsample_idx(len(X_tr), s)
            t = DecisionTreeClassifier(criterion=crit, max_depth=d, random_state=s)
            start = time.time()
            t.fit(X_tr[idx], y_tr[idx])
            fit_t = time.time() - start
            rows.append({'criterion': crit, 'depth': str(d), 'seed': s,
                         'train_acc': accuracy_score(y_tr[idx], t.predict(X_tr[idx])),
                         'val_acc': accuracy_score(y_val, t.predict(X_val)),
                         'time': fit_t, 'leaves': t.get_n_leaves(), 'actual_depth': t.get_depth()})
            if d is None:
                full_trees[(crit, s)] = t
    print(crit, 'done')

df1b = pd.DataFrame(rows)
df1b.to_csv('results/part1b_depth_sweep.csv', index=False)
summ1b = df1b.groupby(['criterion', 'depth']).agg(['mean', 'std'])
summ1b = summ1b.reindex([(c, d) for c in ['gini', 'entropy'] for d in DEPTH_LABELS])
summ1b[['train_acc', 'val_acc', 'time', 'leaves', 'actual_depth']].round(3)

gini done


entropy done


train_acc        val_acc          time        leaves          \
                     mean    std    mean    std   mean    std   mean     std   
criterion depth                                                                
gini      2         0.363  0.001   0.364  0.002  0.317  0.009    4.0   0.000   
          4         0.648  0.003   0.637  0.004  0.627  0.006   16.0   0.000   
          6         0.756  0.003   0.726  0.003  0.942  0.011   59.0   2.000   
          8         0.836  0.006   0.767  0.005  1.249  0.011  158.8   3.962   
          10        0.891  0.004   0.773  0.004  1.526  0.013  308.2  15.156   
          12        0.940  0.004   0.770  0.004  1.753  0.013  482.6  15.534   
          15        0.980  0.003   0.760  0.004  2.019  0.009  685.8   7.362   
          20        0.997  0.001   0.751  0.007  2.256  0.048  811.6  12.740   
          None      1.000  0.000   0.749  0.007  2.339  0.048  845.2  12.418   
entropy   2         0.353  0.011   0.348  0.014  0.349  0.004    4.0   0.000   
          4         0.694  0.005   0.688  0.003  0.743  0.003   16.0   0.000   
          6         0.773  0.007   0.745  0.008  1.175  0.012   62.4   1.673   
          8         0.844  0.006   0.769  0.004  1.570  0.017  184.2   7.463   
          10        0.903  0.005   0.770  0.003  1.876  0.029  357.0  12.669   
          12        0.954  0.003   0.762  0.005  2.087  0.017  536.4  14.876   
          15        0.995  0.002   0.754  0.004  2.190  0.057  691.4  18.050   
          20        1.000  0.000   0.755  0.006  2.188  0.028  714.0  12.826   
          None      1.000  0.000   0.755  0.006  2.165  0.031  714.0  12.826   

                actual_depth         
                        mean    std  
criterion depth                      
gini      2              2.0  0.000  
          4              4.0  0.000  
          6              6.0  0.000  
          8              8.0  0.000  
          10            10.0  0.000  
          12            12.0  0.000  
          15            15.0  0.000  
          20            20.0  0.000  
          None          28.2  3.033  
entropy   2              2.0  0.000  
          4              4.0  0.000  
          6              6.0  0.000  
          8              8.0  0.000  
          10            10.0  0.000  
          12            12.0  0.000  
          15            15.0  0.000  
          20            18.2  1.095  
          None          18.2  1.095

In [8]:
# part 1b - plot
plt.figure(figsize=(7, 4))
x = np.arange(len(DEPTHS))
for crit, c in [('gini', 'C0'), ('entropy', 'C1')]:
    sub = summ1b.loc[crit]
    plt.errorbar(x, sub[('train_acc', 'mean')], yerr=sub[('train_acc', 'std')], color=c, ls='--', marker='o', capsize=3, label=f'{crit} train')
    plt.errorbar(x, sub[('val_acc', 'mean')], yerr=sub[('val_acc', 'std')], color=c, ls='-', marker='s', capsize=3, label=f'{crit} val')
plt.xticks(x, DEPTH_LABELS)
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.title('Fashion-MNIST single tree: accuracy vs depth (mean ± std, 5 subsamples)')
plt.legend()
plt.grid(alpha=0.3)
savefig('p1b_depth_acc')

In [9]:
# part 1b - criterion vs depth
# how big is the criterion effect compared to the depth effect?
val_means = summ1b[('val_acc', 'mean')].unstack(0).reindex(DEPTH_LABELS)
print('val acc range across depths:', (val_means.max() - val_means.min()).round(4).to_dict())
print('largest |gini - entropy| at any depth:', (val_means['gini'] - val_means['entropy']).abs().max().round(4))
print('best depth gini:', val_means['gini'].idxmax(), val_means['gini'].max().round(4))
print('best depth entropy:', val_means['entropy'].idxmax(), val_means['entropy'].max().round(4))

val acc range across depths: {'entropy': 0.4227, 'gini': 0.4093}
largest |gini - entropy| at any depth: 0.0516
best depth gini: 10 0.7732
best depth entropy: 10 0.7705


In [10]:
# part 1b - root features
# do gini and entropy pick the same pixel at the root and at depth 1-2?
# compare node by node following the same left/right path in both trees (same subsample per seed)
def feature_at(tree, path):
    node = 0
    for step in path:
        node = tree.tree_.children_left[node] if step == 'L' else tree.tree_.children_right[node]
        if node == -1:
            return None
    return tree.tree_.feature[node]

PATHS = {0: [''], 1: ['L', 'R'], 2: ['LL', 'LR', 'RL', 'RR']}
agree = {0: [], 1: [], 2: []}
for s in range(N_SEEDS):
    tg, te = full_trees[('gini', s)], full_trees[('entropy', s)]
    for lvl, paths in PATHS.items():
        for pth in paths:
            agree[lvl].append(feature_at(tg, pth) == feature_at(te, pth))
for lvl in agree:
    print(f'depth {lvl}: same feature in {np.sum(agree[lvl])}/{len(agree[lvl])} nodes ({np.mean(agree[lvl]):.0%})')
print('root pixels gini:', [feature_at(full_trees[("gini", s)], "") for s in range(N_SEEDS)])
print('root pixels entropy:', [feature_at(full_trees[("entropy", s)], "") for s in range(N_SEEDS)])

depth 0: same feature in 0/5 nodes (0%)
depth 1: same feature in 1/10 nodes (10%)
depth 2: same feature in 0/20 nodes (0%)
root pixels gini: [np.int64(206), np.int64(206), np.int64(206), np.int64(206), np.int64(206)]
root pixels entropy: [np.int64(94), np.int64(123), np.int64(150), np.int64(94), np.int64(122)]


In [11]:
# part 1c - pruning path
path = DecisionTreeClassifier(random_state=SEED).cost_complexity_pruning_path(X_tr, y_tr)
all_alphas = path.ccp_alphas
print('number of alphas on the path:', len(all_alphas))
# trying all of them would take forever, so we pick ~25 spaced out on a log scale
pos = all_alphas[all_alphas > 0]
alphas = np.geomspace(np.percentile(pos, 5), all_alphas[-3], 25)

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = []
for a in alphas:
    t = DecisionTreeClassifier(ccp_alpha=a, random_state=SEED).fit(X_tr, y_tr)
    cv_scores = cross_val_score(DecisionTreeClassifier(ccp_alpha=a, random_state=SEED), X_tr, y_tr, cv=cv, n_jobs=-1)
    rows.append({'alpha': a, 'val_acc': accuracy_score(y_val, t.predict(X_val)), 'leaves': t.get_n_leaves(),
                 'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std()})
df1c = pd.DataFrame(rows)
df1c.to_csv('results/part1c_pruning.csv', index=False)
best_alpha = df1c.loc[df1c.cv_mean.idxmax(), 'alpha']
print('best alpha by 5-fold cv:', best_alpha)
df1c.round(5)

number of alphas on the path: 783


best alpha by 5-fold cv: 0.0006181140985761197


,alpha,val_acc,leaves,cv_mean,cv_std
0,0.00010,0.7630,843,0.7479,0.00514
1,0.00013,0.7654,832,0.7504,0.00583
2,0.00017,0.7632,715,0.7493,0.00633
3,0.00022,0.7760,514,0.7501,0.00584
4,0.00028,0.7762,431,0.7574,0.00472
5,0.00037,0.7814,317,0.7632,0.00545
6,0.00048,0.7914,221,0.7688,0.00767
7,0.00062,0.7894,150,0.7724,0.00793
8,0.00080,0.7834,106,0.7696,0.00532
9,0.00104,0.7728,75,0.7660,0.00966


In [12]:
# part 1c - plot
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.errorbar(df1c.alpha, df1c.cv_mean, yerr=df1c.cv_std, marker='o', capsize=3, label='5-fold CV acc (train)')
ax1.plot(df1c.alpha, df1c.val_acc, marker='s', label='validation acc')
ax1.axvline(best_alpha, color='gray', ls=':', label=f'chosen α = {best_alpha:.2e}')
ax1.set_xscale('log')
ax1.set_xlabel('ccp_alpha')
ax1.set_ylabel('accuracy')
ax2 = ax1.twinx()
ax2.plot(df1c.alpha, df1c.leaves, color='C3', ls='--', label='number of leaves')
ax2.set_ylabel('number of leaves', color='C3')
ax2.set_yscale('log')
ax1.legend(loc='lower left')
plt.title('Cost-complexity pruning (Gini tree)')
savefig('p1c_pruning')

In [13]:
# part 1c - pruned vs depth
# compare pruned tree vs best depth limited tree (both trained on the full 10k train set)
best_depth_gini = val_means['gini'].idxmax()
best_depth_gini = None if best_depth_gini == 'None' else int(best_depth_gini)
best_depth_ent = val_means['entropy'].idxmax()
best_depth_ent = None if best_depth_ent == 'None' else int(best_depth_ent)

compare = []
for name, params in [('pruned (ccp_alpha)', dict(ccp_alpha=best_alpha)),
                     (f'depth limited (max_depth={best_depth_gini})', dict(max_depth=best_depth_gini)),
                     ('unpruned', dict())]:
    accs, leaves = [], []
    for s in range(N_SEEDS):
        t = DecisionTreeClassifier(random_state=s, **params).fit(X_tr, y_tr)
        accs.append(accuracy_score(y_val, t.predict(X_val)))
        leaves.append(t.get_n_leaves())
    compare.append({'tree': name, 'val_acc': fmt(np.mean(accs), np.std(accs), 4), 'leaves': np.mean(leaves),
                    'val_mean': np.mean(accs)})
compare = pd.DataFrame(compare)
compare

,tree,val_acc,leaves,val_mean
0,pruned (ccp_alpha),0.7884 ± 0.0007,150.0,0.78840
1,depth limited (max_depth=10),0.7821 ± 0.0012,323.0,0.78208
2,unpruned,0.7571 ± 0.0029,1033.6,0.75708


In [14]:
# part 1c - min_samples_leaf sweep
rows = []
for msl in [1, 5, 20, 50]:
    for s in range(N_SEEDS):
        idx = subsample_idx(len(X_tr), s)
        t = DecisionTreeClassifier(min_samples_leaf=msl, random_state=s).fit(X_tr[idx], y_tr[idx])
        rows.append({'min_samples_leaf': msl, 'seed': s, 'train_acc': accuracy_score(y_tr[idx], t.predict(X_tr[idx])),
                     'val_acc': accuracy_score(y_val, t.predict(X_val)), 'leaves': t.get_n_leaves(), 'depth': t.get_depth()})
df_msl = pd.DataFrame(rows)
df_msl.to_csv('results/part1c_min_samples_leaf.csv', index=False)
msl_summ = df_msl.groupby('min_samples_leaf')[['train_acc', 'val_acc', 'leaves', 'depth']].agg(['mean', 'std'])
msl_summ.round(4)

train_acc         val_acc         leaves          depth  \
                      mean     std    mean     std   mean      std  mean   
min_samples_leaf                                                           
1                   1.0000  0.0000  0.7490  0.0069  845.2  12.4177  28.2   
5                   0.9213  0.0013  0.7609  0.0059  461.0  11.7686  20.6   
20                  0.8344  0.0011  0.7654  0.0083  190.8   2.8636  16.2   
50                  0.7906  0.0053  0.7523  0.0087   98.0   0.7071  13.4   

                          
                     std  
min_samples_leaf          
1                 3.0332  
5                 0.8944  
20                1.3038  
50                0.5477

In [15]:
# part 1c - best trees
# pick the best gini tree for later parts: pruned or depth limited, whichever did better on val
if compare.loc[0, 'val_mean'] >= compare.loc[1, 'val_mean']:
    best_gini_params = dict(criterion='gini', ccp_alpha=float(best_alpha))
else:
    best_gini_params = dict(criterion='gini', max_depth=best_depth_gini)
best_ent_params = dict(criterion='entropy', max_depth=best_depth_ent)
print('best gini tree:', best_gini_params)
print('best entropy tree:', best_ent_params)

best gini tree: {'criterion': 'gini', 'ccp_alpha': 0.0006181140985761197}
best entropy tree: {'criterion': 'entropy', 'max_depth': 10}


In [16]:
# part 1d - regression trees
rows = []
for crit in ['squared_error', 'absolute_error', 'friedman_mse']:
    for d in DEPTHS:
        for s in range(N_SEEDS):
            idx = subsample_idx(len(Xc_tr), s)
            t = DecisionTreeRegressor(criterion=crit, max_depth=d, random_state=s)
            start = time.time()
            t.fit(Xc_tr[idx], yc_tr[idx])
            fit_t = time.time() - start
            pred = t.predict(Xc_val)
            rows.append({'criterion': crit, 'depth': str(d), 'seed': s, 'val_rmse': rmse(yc_val, pred),
                         'val_mae': mean_absolute_error(yc_val, pred), 'time': fit_t, 'leaves': t.get_n_leaves()})
df1d = pd.DataFrame(rows)
df1d.to_csv('results/part1d_regression_trees.csv', index=False)
summ1d = df1d.groupby(['criterion', 'depth'])[['val_rmse', 'val_mae', 'time', 'leaves']].agg(['mean', 'std'])
summ1d = summ1d.reindex([(c, d) for c in ['squared_error', 'absolute_error', 'friedman_mse'] for d in DEPTH_LABELS])
summ1d.round(4)

val_rmse         val_mae            time          leaves  \
                         mean     std    mean     std    mean     std    mean   
criterion      depth                                                            
squared_error  2       0.8678  0.0017  0.6550  0.0035  0.0097  0.0004     4.0   
               4       0.7640  0.0045  0.5641  0.0043  0.0184  0.0002    16.0   
               6       0.6957  0.0087  0.5030  0.0088  0.0261  0.0001    62.6   
               8       0.6527  0.0134  0.4531  0.0115  0.0330  0.0002   227.2   
               10      0.6588  0.0222  0.4357  0.0173  0.0395  0.0002   690.4   
               12      0.6844  0.0279  0.4375  0.0175  0.0449  0.0003  1672.4   
               15      0.7132  0.0269  0.4559  0.0179  0.0540  0.0008  4038.0   
               20      0.7300  0.0319  0.4705  0.0204  0.0621  0.0093  7990.8   
               None    0.7355  0.0276  0.4735  0.0173  0.0589  0.0003  9521.0   
absolute_error 2       0.8937  0.0042  0.6347  0.0029  0.0392  0.0009     4.0   
               4       0.7807  0.0052  0.5542  0.0039  0.0742  0.0006    16.0   
               6       0.7145  0.0042  0.4951  0.0060  0.1083  0.0007    63.0   
               8       0.6716  0.0111  0.4514  0.0101  0.1398  0.0008   224.0   
               10      0.6612  0.0100  0.4319  0.0106  0.1669  0.0011   666.0   
               12      0.6847  0.0127  0.4377  0.0141  0.1910  0.0007  1568.0   
               15      0.7174  0.0066  0.4598  0.0099  0.2225  0.0042  3728.0   
               20      0.7480  0.0168  0.4826  0.0145  0.2437  0.0018  7342.4   
               None    0.7533  0.0105  0.4906  0.0106  0.2598  0.0014  9559.0   
friedman_mse   2       0.8678  0.0017  0.6550  0.0035  0.0097  0.0001     4.0   
               4       0.7640  0.0045  0.5641  0.0043  0.0183  0.0002    16.0   
               6       0.6955  0.0088  0.5028  0.0089  0.0266  0.0010    62.6   
               8       0.6523  0.0134  0.4530  0.0115  0.0349  0.0040   227.2   
               10      0.6593  0.0234  0.4360  0.0175  0.0398  0.0003   690.0   
               12      0.6862  0.0255  0.4377  0.0153  0.0451  0.0001  1672.2   
               15      0.7129  0.0309  0.4553  0.0194  0.0534  0.0034  4035.4   
               20      0.7257  0.0270  0.4677  0.0164  0.0578  0.0002  7987.6   
               None    0.7358  0.0287  0.4743  0.0174  0.0594  0.0003  9512.8   

                               
                          std  
criterion      depth           
squared_error  2       0.0000  
               4       0.0000  
               6       1.1402  
               8       5.5408  
               10     33.8866  
               12     77.9923  
               15     88.1164  
               20     90.0983  
               None   13.9821  
absolute_error 2       0.0000  
               4       0.0000  
               6       0.0000  
               8       3.3166  
               10     13.2853  
               12     30.2159  
               15     77.9006  
               20     76.8036  
               None   15.5885  
friedman_mse   2       0.0000  
               4       0.0000  
               6       1.1402  
               8       5.5408  
               10     34.0661  
               12     78.2221  
               15     87.0046  
               20     91.7240  
               None   13.7004

In [17]:
# part 1d - plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(DEPTHS))
for i, metric in enumerate(['val_rmse', 'val_mae']):
    for j, crit in enumerate(['squared_error', 'absolute_error', 'friedman_mse']):
        sub = summ1d.loc[crit]
        axes[i].errorbar(x + (j - 1) * 0.08, sub[(metric, 'mean')], yerr=sub[(metric, 'std')], marker='o', capsize=3, label=crit)
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(DEPTH_LABELS)
    axes[i].set_xlabel('max_depth')
    axes[i].set_ylabel(metric.replace('val_', 'validation ').upper() + ' ($100k)')
    axes[i].grid(alpha=0.3)
    axes[i].legend()
axes[0].set_title('Regression trees: validation RMSE vs depth')
axes[1].set_title('Regression trees: validation MAE vs depth')
savefig('p1d_reg_depth')

In [18]:
# part 1d - best tree
reg_means = summ1d[('val_rmse', 'mean')]
best_crit_reg, best_depth_reg = reg_means.idxmin()
print('best regression tree:', best_crit_reg, best_depth_reg, 'rmse', reg_means.min().round(4))
best_depth_reg = None if best_depth_reg == 'None' else int(best_depth_reg)

# best depth for each criterion separately (needed for the final table)
best_reg_tree_params = {}
for crit in ['squared_error', 'absolute_error', 'friedman_mse']:
    dd = reg_means.loc[crit].idxmin()
    best_reg_tree_params[crit] = dict(criterion=crit, max_depth=None if dd == 'None' else int(dd))
print(best_reg_tree_params)

best_reg_tree = DecisionTreeRegressor(criterion=best_crit_reg, max_depth=best_depth_reg, random_state=SEED).fit(Xc_tr, yc_tr)
plt.figure(figsize=(18, 7))
plot_tree(best_reg_tree, max_depth=2, feature_names=feat_c, filled=True, fontsize=10, precision=2, impurity=False)
plt.title(f'Top 3 levels of the best regression tree ({best_crit_reg}, max_depth={best_depth_reg})')
savefig('p1d_tree_top')

best regression tree: friedman_mse 8 rmse 0.6523
{'squared_error': {'criterion': 'squared_error', 'max_depth': 8}, 'absolute_error': {'criterion': 'absolute_error', 'max_depth': 10}, 'friedman_mse': {'criterion': 'friedman_mse', 'max_depth': 8}}


In [19]:
# part 1d - step plot
# step function: tree trained only on median income
order = np.argsort(Xc_val[:, 0])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(Xc_val[:, 0], yc_val, s=3, alpha=0.25, color='gray', label='actual price (val)')
for d, c in [(2, 'C0'), (4, 'C1'), (8, 'C2')]:
    t1 = DecisionTreeRegressor(max_depth=d, random_state=SEED).fit(Xc_tr[:, [0]], yc_tr)
    axes[0].step(Xc_val[order, 0], t1.predict(Xc_val[order][:, [0]]), where='post', color=c, lw=2, label=f'tree on MedInc only, depth {d}')
axes[0].set_xlabel('median income (MedInc, $10k)')
axes[0].set_ylabel('median house value ($100k)')
axes[0].set_title('Prediction is a step function of the feature')
axes[0].legend(fontsize=8)

pred_best = best_reg_tree.predict(Xc_val)
axes[1].scatter(yc_val, pred_best, s=4, alpha=0.3)
axes[1].plot([0, 5.2], [0, 5.2], 'k--', lw=1)
axes[1].set_xlabel('actual value ($100k)')
axes[1].set_ylabel('predicted value ($100k)')
axes[1].set_title(f'Best tree on validation set (RMSE {rmse(yc_val, pred_best):.3f})')
savefig('p1d_step')

In [20]:
# part 2 - shared bagging setup
FRACS = [0.1, 0.3, 0.5, 0.7, 1.0]

def make_bagging_model(method, frac_rows=1.0, frac_cols=1.0, depth=None, n_trees=100, seed=0, regression=False, oob=False):
    # one function for all 4 methods, the only thing that changes is what gets sampled
    if method == 'pasting':      # rows without replacement, all columns
        bootstrap, ms, mf = False, frac_rows, 1.0
    elif method == 'bagging':    # rows with replacement, all columns
        bootstrap, ms, mf = True, frac_rows, 1.0
    elif method == 'subspaces':  # all rows, subset of columns per tree
        bootstrap, ms, mf = False, 1.0, frac_cols
    elif method == 'patches':    # subset of rows and columns
        bootstrap, ms, mf = False, frac_rows, frac_cols
    if regression:
        return BaggingRegressor(DecisionTreeRegressor(max_depth=depth), n_estimators=n_trees, max_samples=ms, max_features=mf,
                                bootstrap=bootstrap, bootstrap_features=False, oob_score=oob, n_jobs=-1, random_state=seed)
    return BaggingClassifier(DecisionTreeClassifier(max_depth=depth), n_estimators=n_trees, max_samples=ms, max_features=mf,
                             bootstrap=bootstrap, bootstrap_features=False, oob_score=oob, n_jobs=-1, random_state=seed)

def run_one(method, fr, fc, depth, seed):
    m = make_bagging_model(method, fr, fc, depth=depth, seed=seed)
    start = time.time()
    m.fit(X_tr, y_tr)
    return {'method': method, 'rows': fr, 'cols': fc, 'seed': seed, 'time': time.time() - start,
            'val_acc': accuracy_score(y_val, m.predict(X_val))}

def sampling_sweep(depth=None):
    rows = []
    for s in range(3):  # 3 seeds, 5 took way too long
        for f in FRACS:
            rows.append(run_one('pasting', f, 1.0, depth, s))
            rows.append(run_one('bagging', f, 1.0, depth, s))
            rows.append(run_one('subspaces', 1.0, f, depth, s))
            for c in FRACS:
                rows.append(run_one('patches', f, c, depth, s))
        print('seed', s, 'done')
    return pd.DataFrame(rows)

def plot_sampling(df, title, fname, ref_acc=None, ref_label=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for method, col in [('pasting', 'rows'), ('bagging', 'rows'), ('subspaces', 'cols')]:
        g = df[df.method == method].groupby(col)['val_acc'].agg(['mean', 'std'])
        axes[0].errorbar(g.index, g['mean'], yerr=g['std'], marker='o', capsize=3, label=f'{method} (vary {col})')
    if ref_acc is not None:
        axes[0].axhline(ref_acc, color='gray', ls=':', label=ref_label)
    axes[0].set_xlabel('sampled fraction')
    axes[0].set_ylabel('validation accuracy')
    axes[0].set_title(title)
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)
    heat = df[df.method == 'patches'].groupby(['rows', 'cols'])['val_acc'].mean().unstack()
    im = axes[1].imshow(heat.values, origin='lower', cmap='viridis')
    axes[1].set_xticks(range(len(FRACS)))
    axes[1].set_xticklabels(FRACS)
    axes[1].set_yticks(range(len(FRACS)))
    axes[1].set_yticklabels(FRACS)
    axes[1].set_xlabel('column (feature) fraction')
    axes[1].set_ylabel('row (sample) fraction')
    for i in range(len(FRACS)):
        for j in range(len(FRACS)):
            axes[1].text(j, i, f'{heat.values[i, j]:.3f}', ha='center', va='center', color='w', fontsize=8)
    plt.colorbar(im, ax=axes[1], label='validation accuracy')
    axes[1].set_title('Random patches: validation accuracy')
    plt.tight_layout()
    plt.savefig('figs/' + fname + '.png', dpi=130, bbox_inches='tight')
    plt.show()

def best_config(df, method):
    g = df[df.method == method].groupby(['rows', 'cols'])['val_acc'].agg(['mean', 'std'])
    return g['mean'].idxmax(), g.loc[g['mean'].idxmax()]

In [21]:
# part 2a - full trees
df2a = sampling_sweep(depth=None)
df2a.to_csv('results/part2a_sampling_deep.csv', index=False)

single_full_acc = summ1b.loc[('gini', 'None'), ('val_acc', 'mean')]
plot_sampling(df2a, 'Sampling ensembles, 100 full trees', 'p2a_sampling_deep',
              ref_acc=single_full_acc, ref_label='single unpruned tree')

best_frac = {}
for method in ['pasting', 'bagging', 'subspaces', 'patches']:
    (fr, fc), stats = best_config(df2a, method)
    best_frac[method] = (fr, fc)
    print(f'{method:10s} best rows={fr} cols={fc}  val acc {stats["mean"]:.4f} ± {stats["std"]:.4f}')

seed 0 done


seed 1 done


seed 2 done


pasting    best rows=0.5 cols=1.0  val acc 0.8505 ± 0.0008
bagging    best rows=1.0 cols=1.0  val acc 0.8539 ± 0.0046
subspaces  best rows=1.0 cols=0.1  val acc 0.8675 ± 0.0018
patches    best rows=1.0 cols=0.1  val acc 0.8675 ± 0.0018


In [22]:
# part 2a - sweep table
df2a.groupby(['method', 'rows', 'cols'])[['val_acc', 'time']].agg(['mean', 'std']).round(4)

val_acc             time        
                       mean     std     mean     std
method    rows cols                                 
bagging   0.1  1.0   0.8267  0.0017   2.4319  0.0177
          0.3  1.0   0.8477  0.0013   8.1218  0.0738
          0.5  1.0   0.8519  0.0026  13.4625  0.0816
          0.7  1.0   0.8537  0.0011  18.4511  0.0486
          1.0  1.0   0.8539  0.0046  24.4388  0.2066
pasting   0.1  1.0   0.8293  0.0029   2.6538  0.1397
          0.3  1.0   0.8459  0.0018   9.3461  0.0338
          0.5  1.0   0.8505  0.0008  17.3673  0.0404
          0.7  1.0   0.8473  0.0008  26.0148  0.1716
          1.0  1.0   0.7756  0.0011  40.6549  0.2837
patches   0.1  0.1   0.8324  0.0021   0.4921  0.0246
               0.3   0.8329  0.0025   1.0929  0.0314
               0.5   0.8327  0.0010   1.7773  0.1078
               0.7   0.8288  0.0014   2.3464  0.0650
               1.0   0.8293  0.0029   2.6234  0.0973
          0.3  0.1   0.8561  0.0021   1.1782  0.0289
               0.3   0.8526  0.0007   3.1851  0.0831
               0.5   0.8507  0.0016   5.2343  0.0287
               0.7   0.8493  0.0004   7.4278  0.0775
               1.0   0.8459  0.0018   9.4344  0.0929
          0.5  0.1   0.8601  0.0018   1.9251  0.0446
               0.3   0.8569  0.0026   5.5539  0.0624
               0.5   0.8580  0.0024   9.5808  0.3392
               0.7   0.8535  0.0020  13.2385  0.0608
               1.0   0.8505  0.0008  17.3323  0.0325
          0.7  0.1   0.8642  0.0017   2.7360  0.0271
               0.3   0.8621  0.0014   8.1987  0.1088
               0.5   0.8577  0.0027  13.8203  0.1734
               0.7   0.8551  0.0016  19.3998  0.1543
               1.0   0.8473  0.0008  26.0967  0.1070
          1.0  0.1   0.8675  0.0018   4.0014  0.1328
               0.3   0.8649  0.0026  12.1499  0.0701
               0.5   0.8599  0.0044  20.4807  0.0946
               0.7   0.8495  0.0018  29.0175  0.3449
               1.0   0.7756  0.0011  40.5426  0.2143
subspaces 1.0  0.1   0.8675  0.0018   4.0047  0.0420
               0.3   0.8649  0.0026  12.1594  0.1613
               0.5   0.8599  0.0044  20.4959  0.1043
               0.7   0.8495  0.0018  28.9807  0.2356
               1.0   0.7756  0.0011  40.4804  0.2898

In [23]:
# part 2b - oob
val_curves, oob_curves, oob_scores = [], [], []
for s in range(3):
    bag = make_bagging_model('bagging', 1.0, n_trees=300, seed=s, oob=True)
    bag.fit(X_tr, y_tr)
    oob_scores.append(bag.oob_score_)
    # print(len(bag.estimators_samples_[0]))
    # add trees one by one and track accuracy
    val_sum = np.zeros((len(X_val), 10))
    oob_sum = np.zeros((len(X_tr), 10))
    va, oo = [], []
    for est, samp, feats in zip(bag.estimators_, bag.estimators_samples_, bag.estimators_features_):
        val_sum += est.predict_proba(X_val[:, feats])
        oob_mask = np.ones(len(X_tr), dtype=bool)
        oob_mask[samp] = False  # rows not in this tree's bootstrap sample
        oob_sum[oob_mask] += est.predict_proba(X_tr[oob_mask][:, feats])
        va.append(np.mean(val_sum.argmax(1) == y_val))
        has_vote = oob_sum.sum(1) > 0
        oo.append(np.mean(oob_sum[has_vote].argmax(1) == y_tr[has_vote]))
    val_curves.append(va)
    oob_curves.append(oo)
    print(f'seed {s}: sklearn oob_score_={bag.oob_score_:.4f}  my oob at 300 trees={oo[-1]:.4f}  val={va[-1]:.4f}')

val_curves, oob_curves = np.array(val_curves), np.array(oob_curves)
n_axis = np.arange(1, 301)
plt.figure(figsize=(7, 4))
for curves, lab in [(val_curves, 'validation accuracy'), (oob_curves, 'OOB accuracy')]:
    m, sd = curves.mean(0), curves.std(0)
    plt.plot(n_axis, m, label=lab)
    plt.fill_between(n_axis, m - sd, m + sd, alpha=0.25)
plt.xscale('log')
plt.xlabel('number of trees')
plt.ylabel('accuracy')
plt.title(f'Bagging (full trees): accuracy vs ensemble size (mean ± std, 3 seeds)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figs/p2b_oob.png', dpi=130, bbox_inches='tight')
plt.show()

for k in [1, 10, 50, 100, 300]:
    print(f'{k:4d} trees: val {val_curves[:, k-1].mean():.4f}  oob {oob_curves[:, k-1].mean():.4f}  gap {val_curves[:, k-1].mean() - oob_curves[:, k-1].mean():+.4f}')

seed 0: sklearn oob_score_=0.8483  my oob at 300 trees=0.8483  val=0.8562


seed 1: sklearn oob_score_=0.8471  my oob at 300 trees=0.8471  val=0.8556


seed 2: sklearn oob_score_=0.8481  my oob at 300 trees=0.8481  val=0.8546
   1 trees: val 0.7377  oob 0.7295  gap +0.0081
  10 trees: val 0.8282  oob 0.7835  gap +0.0447
  50 trees: val 0.8513  oob 0.8384  gap +0.0129
 100 trees: val 0.8539  oob 0.8432  gap +0.0108
 300 trees: val 0.8555  oob 0.8478  gap +0.0076


In [24]:
# part 2c - bias variance
def bias_variance(make_model, n_runs=10):
    preds = []
    for b in range(n_runs):
        rng = np.random.RandomState(1000 + b)
        idx = rng.choice(len(X_tr), len(X_tr), replace=True)
        m = make_model(b)
        m.fit(X_tr[idx], y_tr[idx])
        preds.append(m.predict(X_val))
    preds = np.array(preds)                                    # n_runs x n_val
    counts = np.array([(preds == k).sum(0) for k in range(10)])  # votes per class for each point
    majority = counts.argmax(0)
    return {'disagreement (variance)': np.mean(1 - counts.max(0) / n_runs),   # avg fraction of models disagreeing with the majority
            'majority vote error (bias)': np.mean(majority != y_val),
            'avg single model error': np.mean(preds != y_val[None, :])}

bv_models = {
    'single unpruned tree': lambda s: DecisionTreeClassifier(random_state=s),
    'single depth-4 tree': lambda s: DecisionTreeClassifier(max_depth=4, random_state=s),
}
for method in ['pasting', 'bagging', 'subspaces', 'patches']:
    fr, fc = best_frac[method]
    bv_models[method] = lambda s, method=method, fr=fr, fc=fc: make_bagging_model(method, fr, fc, seed=s)

rows = []
for name, mk in bv_models.items():
    r = bias_variance(mk, n_runs=10)
    r['model'] = name
    rows.append(r)
    print(name, 'done')
df2c = pd.DataFrame(rows).set_index('model')
df2c.to_csv('results/part2c_bias_variance.csv')
df2c.round(4)

single unpruned tree done


single depth-4 tree done


pasting done


bagging done


subspaces done


patches done


,disagreement (variance),majority vote error (bias),avg single model error
model,,,
single unpruned tree,0.2083,0.1680,0.2645
single depth-4 tree,0.0911,0.3372,0.3637
pasting,0.0607,0.1446,0.1612
bagging,0.0621,0.1474,0.1615
subspaces,0.0432,0.1332,0.1421
patches,0.0432,0.1332,0.1421


In [25]:
# part 2c - plot
ax = df2c.plot.bar(figsize=(9, 4), rot=20)
ax.set_ylabel('fraction of validation points')
ax.set_xlabel('')
ax.set_title('Bias–variance over 10 bootstrap resamples of the training set')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('figs/p2c_bias_variance.png', dpi=130, bbox_inches='tight')
plt.show()

In [26]:
# part 2d - depth 4 trees
df2d = sampling_sweep(depth=4)
df2d.to_csv('results/part2d_sampling_depth4.csv', index=False)
single_d4_acc = summ1b.loc[('gini', '4'), ('val_acc', 'mean')]
plot_sampling(df2d, 'Sampling ensembles, 100 depth-4 trees', 'p2d_sampling_depth4',
              ref_acc=single_d4_acc, ref_label='single depth-4 tree')

rows = []
for method in ['pasting', 'bagging', 'subspaces', 'patches']:
    _, deep = best_config(df2a, method)
    _, shallow = best_config(df2d, method)
    rows.append({'method': method, 'best deep acc': deep['mean'], 'gain over single deep tree': deep['mean'] - single_full_acc,
                 'best depth-4 acc': shallow['mean'], 'gain over single depth-4 tree': shallow['mean'] - single_d4_acc})
pd.DataFrame(rows).round(4)

seed 0 done


seed 1 done


seed 2 done


,method,best deep acc,gain over single deep tree,best depth-4 acc,gain over single depth-4 tree
0,pasting,0.8505,0.1014,0.7343,0.0976
1,bagging,0.8539,0.1049,0.7342,0.0976
2,subspaces,0.8675,0.1185,0.7449,0.1082
3,patches,0.8675,0.1185,0.7476,0.1110


In [27]:
# part 3a - max_features sweep
p = X_tr.shape[1]
MF_OPTIONS = {'1': 1, 'log2(p)': 'log2', 'sqrt(p)': 'sqrt', '0.1p': 0.1, '0.3p': 0.3, 'p (=bagging)': None}
mf_nfeat = {'1': 1, 'log2(p)': int(np.log2(p)), 'sqrt(p)': int(np.sqrt(p)), '0.1p': int(0.1 * p), '0.3p': int(0.3 * p), 'p (=bagging)': p}

rows = []
for crit in ['gini', 'entropy']:
    for label, mf in MF_OPTIONS.items():
        for s in range(3):
            # rf = RandomForestClassifier(n_estimators=100, max_features=mf, n_jobs=-1)  # tried 100 trees first
            rf = RandomForestClassifier(n_estimators=300, max_features=mf, criterion=crit, n_jobs=-1, random_state=s)
            start = time.time()
            rf.fit(X_tr, y_tr)
            rows.append({'criterion': crit, 'max_features': label, 'n_features': mf_nfeat[label], 'seed': s,
                         'time': time.time() - start, 'val_acc': accuracy_score(y_val, rf.predict(X_val))})
        print(crit, label, 'done')
df3a = pd.DataFrame(rows)
df3a.to_csv('results/part3a_max_features.csv', index=False)
summ3a = df3a.groupby(['criterion', 'max_features', 'n_features']).agg(
    val_mean=('val_acc', 'mean'), val_std=('val_acc', 'std'), time_mean=('time', 'mean'), time_std=('time', 'std')).reset_index()
summ3a = summ3a.sort_values(['criterion', 'n_features'])
summ3a.round(4)

gini 1 done


gini log2(p) done


gini sqrt(p) done


gini 0.1p done


gini 0.3p done


...

entropy sqrt(p) done


entropy 0.1p done


entropy 0.3p done


entropy p (=bagging) done


,criterion,max_features,n_features,val_mean,val_std,time_mean,time_std
2,entropy,1,1,0.8397,0.0025,0.4933,0.0491
3,entropy,log2(p),9,0.8611,0.0021,4.7605,2.0672
5,entropy,sqrt(p),28,0.8668,0.0011,179.6331,166.5966
0,entropy,0.1p,78,0.8695,0.0011,10.0071,4.2746
1,entropy,0.3p,235,0.8656,0.0007,79.6338,63.4638
4,entropy,p (=bagging),784,0.8589,0.0006,162.2067,56.6357
8,gini,1,1,0.8415,0.0015,0.6454,0.1026
9,gini,log2(p),9,0.8599,0.0009,1.7262,0.0688
11,gini,sqrt(p),28,0.8645,0.0025,4.4859,0.0233
6,gini,0.1p,78,0.8670,0.0012,12.7246,0.1495


In [28]:
# part 3a - plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for crit in ['gini', 'entropy']:
    sub = summ3a[summ3a.criterion == crit]
    axes[0].errorbar(sub.n_features, sub.val_mean, yerr=sub.val_std, marker='o', capsize=3, label=crit)
    axes[1].errorbar(sub.n_features, sub.time_mean, yerr=sub.time_std, marker='o', capsize=3, label=crit)
for ax in axes:
    ax.set_xscale('log')
    ax.set_xlabel('max_features (number of pixels tried per split)')
    ax.grid(alpha=0.3)
    ax.legend()
sub = summ3a[summ3a.criterion == 'gini']
for lab, nf, acc in zip(sub.max_features, sub.n_features, sub.val_mean):
    axes[0].annotate(lab, (nf, acc), fontsize=7, textcoords='offset points', xytext=(3, -10))
axes[0].set_ylabel('validation accuracy')
axes[0].set_title('Random forest (300 trees): accuracy vs max_features')
axes[1].set_ylabel('training time (s)')
axes[1].set_title('Random forest: training time vs max_features')
plt.tight_layout()
plt.savefig('figs/p3a_max_features.png', dpi=130, bbox_inches='tight')
plt.show()

In [29]:
# part 3a - best rf
g3a = df3a.groupby(['criterion', 'max_features'])['val_acc'].mean()
best_rf_crit, best_rf_mf_label = g3a.idxmax()
best_rf_params = dict(criterion=best_rf_crit, max_features=MF_OPTIONS[best_rf_mf_label])
print('best rf:', best_rf_params, 'val acc', g3a.max().round(4))

best rf: {'criterion': 'entropy', 'max_features': 0.1} val acc 0.8695


In [30]:
# part 3b - rf vs extra trees
rows = []
rf_models, et_models = [], []
for s in range(N_SEEDS):
    for name, Cls in [('random forest', RandomForestClassifier), ('extra trees', ExtraTreesClassifier)]:
        m = Cls(n_estimators=300, n_jobs=-1, random_state=s, **best_rf_params)
        start = time.time()
        m.fit(X_tr, y_tr)
        t = time.time() - start
        rows.append({'model': name, 'seed': s, 'time': t, 'val_acc': accuracy_score(y_val, m.predict(X_val)),
                     'nodes': count_nodes(m)})
        if s == 0:
            (rf_models if name == 'random forest' else et_models).append(m)
df3b = pd.DataFrame(rows)
df3b.to_csv('results/part3b_rf_vs_et.csv', index=False)
summ3b = df3b.groupby('model')[['val_acc', 'time', 'nodes']].agg(['mean', 'std'])
summ3b[('val_acc', 'var')] = df3b.groupby('model')['val_acc'].var()
summ3b.round(5)

val_acc              time               nodes              \
                  mean      std     mean      std       mean         std   
model                                                                      
extra trees    0.86912  0.00199  2.41458  0.47211  1022132.4  1212.53981   
random forest  0.86924  0.00086  7.65987  0.26325   466039.6   491.23396   

              val_acc  
                  var  
model                  
extra trees       0.0  
random forest     0.0

In [31]:
# part 3b - extra trees max_features
# also sweep max_features for extra trees with 1 seed, since the best value can be different from rf
rows = []
for label, mf in MF_OPTIONS.items():
    et = ExtraTreesClassifier(n_estimators=300, max_features=mf, criterion=best_rf_crit, n_jobs=-1, random_state=0)
    start = time.time()
    et.fit(X_tr, y_tr)
    rows.append({'max_features': label, 'val_acc': accuracy_score(y_val, et.predict(X_val)), 'time': time.time() - start})
df3b_et = pd.DataFrame(rows)
best_et_mf_label = df3b_et.loc[df3b_et.val_acc.idxmax(), 'max_features']
best_et_params = dict(criterion=best_rf_crit, max_features=MF_OPTIONS[best_et_mf_label])
print('best extra trees:', best_et_params)
df3b_et.round(4)

best extra trees: {'criterion': 'entropy', 'max_features': None}


,max_features,val_acc,time
0,1,0.8352,0.5111
1,log2(p),0.8586,0.7479
2,sqrt(p),0.8654,1.1106
3,0.1p,0.8660,2.2173
4,0.3p,0.8712,6.7655
5,p (=bagging),0.8722,17.7836


In [32]:
# part 3c - importance fashion
rf0 = rf_models[0]
imp_impurity = rf0.feature_importances_
rf0.set_params(n_jobs=1)  # permutation_importance already runs in parallel, avoid nesting
perm = permutation_importance(rf0, X_val, y_val, n_repeats=3, random_state=SEED, n_jobs=-1)
rf0.set_params(n_jobs=-1)
imp_perm = perm.importances_mean

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
im = axes[0].imshow(imp_impurity.reshape(28, 28), cmap='hot')
axes[0].set_title('Impurity (MDI) importance')
plt.colorbar(im, ax=axes[0], fraction=0.046)
im = axes[1].imshow(imp_perm.reshape(28, 28), cmap='hot')
axes[1].set_title('Permutation importance (val, drop in acc)')
plt.colorbar(im, ax=axes[1], fraction=0.046)
axes[2].scatter(imp_impurity, imp_perm, s=5)
axes[2].set_xlabel('impurity importance')
axes[2].set_ylabel('permutation importance')
axes[2].set_title(f'Per-pixel comparison (Spearman r = {pd.Series(imp_impurity).corr(pd.Series(imp_perm), method="spearman"):.2f})')
for ax in axes[:2]:
    ax.set_xlabel('pixel column')
    ax.set_ylabel('pixel row')
plt.tight_layout()
plt.savefig('figs/p3c_importance_fashion.png', dpi=130, bbox_inches='tight')
plt.show()
print('pixels with negative permutation importance:', np.sum(imp_perm < 0))
print('top-20 overlap between the two methods:', len(set(np.argsort(imp_impurity)[-20:]) & set(np.argsort(imp_perm)[-20:])))

pixels with negative permutation importance: 298
top-20 overlap between the two methods: 1


In [33]:
# part 3c - importance california
# california: also add two fake features to show the cardinality bias
# random_noise is continuous (lots of unique values), random_coin is 0/1. neither has anything to do with price
rng = np.random.RandomState(SEED)
Xc_tr_noise = np.column_stack([Xc_tr, rng.randn(len(Xc_tr)), rng.randint(0, 2, len(Xc_tr))])
Xc_val_noise = np.column_stack([Xc_val, rng.randn(len(Xc_val)), rng.randint(0, 2, len(Xc_val))])
names_noise = feat_c + ['random_noise', 'random_coin']

rfc = RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=SEED).fit(Xc_tr_noise, yc_tr)
perm_c = permutation_importance(rfc, Xc_val_noise, yc_val, n_repeats=10, random_state=SEED)

imp_df = pd.DataFrame({'impurity (MDI)': rfc.feature_importances_,
                       'permutation (val)': perm_c.importances_mean,
                       'perm std': perm_c.importances_std}, index=names_noise)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
imp_df['impurity (MDI)'].sort_values().plot.barh(ax=axes[0], color='C0')
axes[0].set_title('California RF: impurity importance')
axes[0].set_xlabel('mean decrease in impurity (normalized)')
srt = imp_df.sort_values('permutation (val)')
axes[1].barh(srt.index, srt['permutation (val)'], xerr=srt['perm std'], color='C1')
axes[1].set_title('California RF: permutation importance (validation)')
axes[1].set_xlabel('drop in R² when shuffled')
plt.tight_layout()
plt.savefig('figs/p3c_importance_california.png', dpi=130, bbox_inches='tight')
plt.show()
imp_df.round(4)

,impurity (MDI),permutation (val),perm std
MedInc,0.5216,0.7950,0.0211
HouseAge,0.0522,0.0610,0.0054
AveRooms,0.0388,0.0188,0.0009
AveBedrms,0.0270,0.0076,0.0011
Population,0.0283,0.0064,0.0006
AveOccup,0.1350,0.1841,0.0098
Latitude,0.0891,0.4032,0.0131
Longitude,0.0873,0.3047,0.0038
random_noise,0.0182,0.0003,0.0005
random_coin,0.0024,0.0002,0.0002


In [34]:
# part 4a - adaboost
# sklearn 1.8 only does SAMME now so no algorithm arg. adaboost is single threaded
# so the 12 configs run in parallel. its basically deterministic so 1 run per config
def run_ada(depth, lr):
    ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=depth), n_estimators=500, learning_rate=lr, random_state=SEED)
    start = time.time()
    ada.fit(X_tr, y_tr)
    t = time.time() - start
    tr_err = [1 - np.mean(pr == y_tr) for pr in ada.staged_predict(X_tr)]
    va_err = [1 - np.mean(pr == y_val) for pr in ada.staged_predict(X_val)]
    return {'depth': depth, 'lr': lr, 'train_err': tr_err, 'val_err': va_err, 'time': t, 'rounds_fit': len(ada.estimators_)}

ADA_DEPTHS = [1, 2, 4, 8]
ADA_LRS = [0.1, 0.5, 1.0]
# ada_results = [run_ada(d, lr) for d in ADA_DEPTHS for lr in ADA_LRS]  # way too slow one at a time
ada_results = Parallel(n_jobs=-1)(delayed(run_ada)(d, lr) for d in ADA_DEPTHS for lr in ADA_LRS)

rows = []
for r in ada_results:
    best_round = int(np.argmin(r['val_err'])) + 1
    zero_round = next((i + 1 for i, e in enumerate(r['train_err']) if e == 0), None)
    rows.append({'depth': r['depth'], 'lr': r['lr'], 'rounds_fit': r['rounds_fit'], 'time': r['time'],
                 'final_train_err': r['train_err'][-1], 'final_val_err': r['val_err'][-1],
                 'best_val_err': min(r['val_err']), 'best_round': best_round, 'train_err_zero_at': zero_round})
df4a = pd.DataFrame(rows)
df4a.to_csv('results/part4a_adaboost.csv', index=False)
df4a.round(4)

,depth,lr,rounds_fit,time,final_train_err,final_val_err,best_val_err,best_round,train_err_zero_at
0,1,0.1,500,474.7126,0.5076,0.5120,0.4994,164,NaN
1,1,0.5,500,470.7225,0.3999,0.4068,0.4030,498,NaN
2,1,1.0,500,469.3895,0.3598,0.3680,0.3308,187,NaN
3,2,0.1,500,1060.2533,0.3001,0.3116,0.2612,154,NaN
4,2,0.5,500,1068.4814,0.2614,0.2838,0.2674,58,NaN
5,2,1.0,500,1064.3761,0.2509,0.2712,0.2514,408,NaN
6,4,0.1,500,4657.5130,0.1349,0.1880,0.1804,408,NaN
7,4,0.5,500,4669.9349,0.1172,0.1946,0.1896,374,NaN
8,4,1.0,500,4664.5602,0.1085,0.1978,0.1956,406,NaN
9,8,0.1,500,5111.0774,0.0000,0.1292,0.1282,443,147.0


In [35]:
# part 4a - plot
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2), sharey=True)
for i, d in enumerate(ADA_DEPTHS):
    for j, lr in enumerate(ADA_LRS):
        r = [r for r in ada_results if r['depth'] == d and r['lr'] == lr][0]
        rounds = np.arange(1, len(r['val_err']) + 1)
        axes[i].plot(rounds, r['train_err'], ls='--', color=f'C{j}', label=f'train, lr={lr}')
        axes[i].plot(rounds, r['val_err'], ls='-', color=f'C{j}', label=f'val, lr={lr}')
    axes[i].set_title(f'AdaBoost, base tree depth {d}')
    axes[i].set_xlabel('boosting round')
    axes[i].grid(alpha=0.3)
axes[0].set_ylabel('error rate')
axes[0].legend(fontsize=7)
savefig('p4a_adaboost_curves')

In [36]:
# part 4a - best adaboost
best_row = df4a.loc[df4a.best_val_err.idxmin()]
best_ada = dict(depth=int(best_row.depth), lr=float(best_row.lr), rounds=int(best_row.best_round))
print('best adaboost:', best_ada, 'val acc', 1 - best_row.best_val_err)

def make_ada(seed, n_rounds=None, cfg=None):
    cfg = best_ada if cfg is None else cfg
    return AdaBoostClassifier(DecisionTreeClassifier(max_depth=cfg['depth']), n_estimators=n_rounds or cfg['rounds'],
                              learning_rate=cfg['lr'], random_state=seed)

best adaboost: {'depth': 8, 'lr': 0.1, 'rounds': 443} val acc 0.8718


In [37]:
# part 4b - sample weights
# sklearn doesnt save the sample weights so we replay the SAMME update:
# w = w * exp(alpha * wrong), then normalize
def replay_weights(ada, X, y):
    w = np.ones(len(y)) / len(y)
    for est, alpha in zip(ada.estimators_, ada.estimator_weights_):
        wrong = est.predict(X) != y
        w = w * np.exp(alpha * wrong)
        w = w / w.sum()
    return w

ada50 = make_ada(SEED, n_rounds=50).fit(X_tr, y_tr)
w50 = replay_weights(ada50, X_tr, y_tr)
pred50 = ada50.predict(X_tr)
top20 = np.argsort(w50)[::-1][:20]

fig, axes = plt.subplots(2, 10, figsize=(16, 4.5))
for ax, i in zip(axes.ravel(), top20):
    ax.imshow(X_tr[i].reshape(28, 28), cmap='gray_r')
    ax.set_title(f'label: {CLASS_NAMES[y_tr[i]]}\npred: {CLASS_NAMES[pred50[i]]}\nw={w50[i] * len(w50):.0f}× avg', fontsize=7)
    ax.axis('off')
plt.suptitle(f'20 highest-weight training images after 50 AdaBoost rounds (depth {best_ada["depth"]}, lr {best_ada["lr"]})')
savefig('p4b_top_weights')
print('weight share of the top 1% of points:', np.sort(w50)[::-1][:len(w50) // 100].sum().round(3))
print('labels of top 20:', [CLASS_NAMES[k] for k in y_tr[top20]])

weight share of the top 1% of points: 0.156
labels of top 20: ['Pullover', 'Shirt', 'Pullover', 'T-shirt/top', 'Pullover', 'Pullover', 'Dress', 'Bag', 'Trouser', 'Pullover', 'Shirt', 'Shirt', 'Shirt', 'Pullover', 'Shirt', 'Shirt', 'T-shirt/top', 'Bag', 'T-shirt/top', 'Trouser']


In [38]:
# part 4b - label noise
# label noise: flip 10% of the training labels to a different random class
def flip_labels(y, frac, seed):
    rng = np.random.RandomState(seed)
    y2 = y.copy()
    idx = rng.choice(len(y), int(frac * len(y)), replace=False)
    for i in idx:
        y2[i] = rng.choice([k for k in range(10) if k != y[i]])
    return y2, idx

def noise_run(model_name, noisy, s):
    y_use = flip_labels(y_tr, 0.1, s)[0] if noisy else y_tr
    if model_name == 'AdaBoost':
        m = make_ada(s)
    else:
        m = RandomForestClassifier(n_estimators=300, n_jobs=1, random_state=s, **best_rf_params)
    m.fit(X_tr, y_use)
    return {'model': model_name, 'labels': 'noisy (10% flipped)' if noisy else 'clean', 'seed': s,
            'val_acc': accuracy_score(y_val, m.predict(X_val))}

rows = Parallel(n_jobs=-1)(delayed(noise_run)(mn, nz, s) for mn in ['AdaBoost', 'Random forest']
                           for nz in [False, True] for s in range(3))
df4b = pd.DataFrame(rows)
df4b.to_csv('results/part4b_label_noise.csv', index=False)
noise_summ = df4b.groupby(['model', 'labels'])['val_acc'].agg(['mean', 'std']).unstack()
noise_summ[('drop', '')] = noise_summ[('mean', 'clean')] - noise_summ[('mean', 'noisy (10% flipped)')]
noise_summ.round(4)

mean                         std                        drop
labels          clean noisy (10% flipped)   clean noisy (10% flipped)        
model                                                                        
AdaBoost       0.8701              0.8522  0.0013               0.002  0.0179
Random forest  0.8695              0.8663  0.0011               0.001  0.0032

In [39]:
# part 4b - noisy weights
# where does adaboost put its weight when labels are noisy?
y_noisy, flipped = flip_labels(y_tr, 0.1, 0)
ada50n = make_ada(SEED, n_rounds=50).fit(X_tr, y_noisy)
wn = replay_weights(ada50n, X_tr, y_noisy)
is_flipped = np.zeros(len(y_tr), bool)
is_flipped[flipped] = True
top = np.argsort(wn)[::-1][:len(wn) // 20]  # top 5% by weight
print(f'flipped labels make up {is_flipped.mean():.0%} of the data but {is_flipped[top].mean():.0%} of the top-5% weighted points')
print(f'total weight on flipped points: {wn[is_flipped].sum():.2f}')

flipped labels make up 10% of the data but 90% of the top-5% weighted points
total weight on flipped points: 0.53


In [40]:
# part 4c - hist gradient boosting
# early stopping uses 10% of the TRAIN set (validation_fraction), our val set is only
# used to compare configs
rows = []
hgb_curves = {}
for lr in [0.05, 0.1, 0.3]:
    for depth in [3, None]:
        hgb = HistGradientBoostingClassifier(learning_rate=lr, max_depth=depth, max_iter=300, early_stopping=True,
                                             validation_fraction=0.1, n_iter_no_change=10, random_state=SEED)
        start = time.time()
        hgb.fit(X_tr, y_tr)
        t = time.time() - start
        hgb_curves[(lr, depth)] = -hgb.validation_score_   # log loss on the internal held-out set per iteration
        rows.append({'lr': lr, 'max_depth': str(depth), 'n_iter': hgb.n_iter_, 'time': t,
                     'val_acc': accuracy_score(y_val, hgb.predict(X_val))})
        print(lr, depth, 'done', round(t, 1))
df4c = pd.DataFrame(rows)
df4c.to_csv('results/part4c_hgb.csv', index=False)
best_hgb_row = df4c.loc[df4c.val_acc.idxmax()]
best_hgb_params = dict(learning_rate=float(best_hgb_row.lr),
                       max_depth=None if best_hgb_row.max_depth == 'None' else int(best_hgb_row.max_depth),
                       max_iter=300, early_stopping=True, validation_fraction=0.1, n_iter_no_change=10)
print('best hgb:', best_hgb_params)
df4c.round(4)

0.05 3 done 38.8


0.05 None done 32.3


0.1 3 done 16.8


0.1 None done 25.3


0.3 3 done 3.4


0.3 None done 14.0
best hgb: {'learning_rate': 0.05, 'max_depth': None, 'max_iter': 300, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 10}


,lr,max_depth,n_iter,time,val_acc
0,0.05,3,254,38.8174,0.8660
1,0.05,None,89,32.3133,0.8734
2,0.10,3,133,16.7974,0.8674
3,0.10,None,53,25.3331,0.8698
4,0.30,3,19,3.4141,0.8134
5,0.30,None,25,13.9719,0.8660


In [41]:
# part 4c - hgb plot
fig, ax = plt.subplots(figsize=(7, 4))
for (lr, depth), curve in hgb_curves.items():
    if str(depth) == best_hgb_row.max_depth:
        ax.plot(np.arange(len(curve)), curve, label=f'lr={lr} (stopped at {len(curve) - 1})')
ax.set_xlabel('boosting iteration')
ax.set_ylabel('log loss on internal early-stopping set')
ax.set_title(f'HistGradientBoosting on Fashion-MNIST (max_depth={best_hgb_row.max_depth})')
ax.legend()
ax.grid(alpha=0.3)
savefig('p4c_hgb_curves')

In [42]:
# part 4c - gbr
# gradient boosting regressor on california: tune lr x depth with early stopping
rows = []
for lr in [0.03, 0.1, 0.3]:
    for depth in [3, 5, 7]:
        gbr = GradientBoostingRegressor(learning_rate=lr, max_depth=depth, n_estimators=2000, n_iter_no_change=20,
                                        validation_fraction=0.1, random_state=SEED)
        start = time.time()
        gbr.fit(Xc_tr, yc_tr)
        pr = gbr.predict(Xc_val)
        rows.append({'lr': lr, 'depth': depth, 'rounds_used': gbr.n_estimators_, 'time': time.time() - start,
                     'val_rmse': np.sqrt(mean_squared_error(yc_val, pr)), 'val_mae': mean_absolute_error(yc_val, pr)})
df4c_gbr = pd.DataFrame(rows)
df4c_gbr.to_csv('results/part4c_gbr.csv', index=False)
best_gbr_row = df4c_gbr.loc[df4c_gbr.val_rmse.idxmin()]
best_gbr_params = dict(learning_rate=float(best_gbr_row.lr), max_depth=int(best_gbr_row.depth), n_estimators=2000,
                       n_iter_no_change=20, validation_fraction=0.1)
print('best gbr:', best_gbr_params)
df4c_gbr.round(4)

best gbr: {'learning_rate': 0.03, 'max_depth': 7, 'n_estimators': 2000, 'n_iter_no_change': 20, 'validation_fraction': 0.1}


,lr,depth,rounds_used,time,val_rmse,val_mae
0,0.03,3,1222,20.1125,0.4941,0.3383
1,0.03,5,949,25.3951,0.4804,0.3200
2,0.03,7,530,20.0032,0.4731,0.3133
3,0.10,3,645,10.8767,0.4869,0.3289
4,0.10,5,446,12.1382,0.4793,0.3194
5,0.10,7,290,10.6916,0.4742,0.3119
6,0.30,3,185,3.0732,0.4967,0.3369
7,0.30,5,101,2.6993,0.4893,0.3291
8,0.30,7,129,4.7521,0.4986,0.3320


In [43]:
# part 4c - lr vs rounds
# learning rate vs number of rounds trade-off (no early stopping here so we see the whole curve)
plt.figure(figsize=(7, 4.2))
lr_tradeoff = []
for lr in [0.01, 0.03, 0.1, 0.3, 1.0]:
    gbr = GradientBoostingRegressor(learning_rate=lr, max_depth=int(best_gbr_row.depth), n_estimators=1000,
                                    random_state=SEED).fit(Xc_tr, yc_tr)
    curve = []
    for pr in gbr.staged_predict(Xc_val):
        curve.append(np.sqrt(mean_squared_error(yc_val, pr)))
    k = int(np.argmin(curve))
    lr_tradeoff.append({'lr': lr, 'best_round': k + 1, 'best_val_rmse': curve[k]})
    line, = plt.plot(np.arange(1, len(curve) + 1), curve, label=f'lr={lr}')
    plt.scatter([k + 1], [curve[k]], color=line.get_color(), zorder=5)
plt.xscale('log')
plt.ylim(0.4, 1.2)
plt.xlabel('number of boosting rounds')
plt.ylabel('validation RMSE ($100k)')
plt.title(f'GradientBoostingRegressor: learning rate vs rounds (depth {int(best_gbr_row.depth)}, dot = minimum)')
plt.legend()
plt.grid(alpha=0.3)
savefig('p4c_lr_vs_rounds')
pd.DataFrame(lr_tradeoff).round(4)

,lr,best_round,best_val_rmse
0,0.01,1000,0.4756
1,0.03,1000,0.4678
2,0.10,995,0.4701
3,0.30,73,0.4968
4,1.00,4,0.5958


In [44]:
# part 4c - adaboost.r2 for comparison
rows = []
for depth in [3, 6, 10]:
    for lr in [0.1, 0.5, 1.0]:
        adr = AdaBoostRegressor(DecisionTreeRegressor(max_depth=depth), n_estimators=200, learning_rate=lr,
                                random_state=SEED)
        start = time.time()
        adr.fit(Xc_tr, yc_tr)
        pr = adr.predict(Xc_val)
        rows.append({'depth': depth, 'lr': lr, 'time': time.time() - start, 'val_rmse': np.sqrt(mean_squared_error(yc_val, pr)),
                     'val_mae': mean_absolute_error(yc_val, pr)})
df4c_adr = pd.DataFrame(rows)
df4c_adr.to_csv('results/part4c_adaboost_r2.csv', index=False)
best_adr_row = df4c_adr.loc[df4c_adr.val_rmse.idxmin()]
best_adr_params = dict(depth=int(best_adr_row.depth), lr=float(best_adr_row.lr))
print('best adaboost.r2:', best_adr_params)
print(f"best GBR      val rmse {best_gbr_row.val_rmse:.4f}  mae {best_gbr_row.val_mae:.4f}  time {best_gbr_row.time:.1f}s")
print(f"best AdaB.R2  val rmse {best_adr_row.val_rmse:.4f}  mae {best_adr_row.val_mae:.4f}  time {best_adr_row.time:.1f}s")
df4c_adr.round(4)

best adaboost.r2: {'depth': 10, 'lr': 1.0}
best GBR      val rmse 0.4731  mae 0.3133  time 20.0s
best AdaB.R2  val rmse 0.4987  mae 0.3673  time 6.4s


,depth,lr,time,val_rmse,val_mae
0,3,0.1,2.2258,0.7535,0.6144
1,3,0.5,1.0394,0.8888,0.7720
2,3,1.0,0.5903,0.9015,0.7877
3,6,0.1,5.3616,0.6252,0.5021
4,6,0.5,3.6836,0.7166,0.6114
5,6,1.0,3.1498,0.7261,0.6216
6,10,0.1,8.5570,0.5108,0.3548
7,10,0.5,6.9788,0.5053,0.3689
8,10,1.0,6.4307,0.4987,0.3673


In [45]:
# part 5 - voting and stacking
def make_members(seed):
    return [('tree', DecisionTreeClassifier(random_state=seed, **best_gini_params)),
            ('rf', RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=seed, **best_rf_params)),
            ('ada', make_ada(seed)),
            ('hgb', HistGradientBoostingClassifier(random_state=seed, **best_hgb_params))]

start = time.time()
vote = VotingClassifier(make_members(SEED), voting='soft', n_jobs=-1).fit(X_tr, y_tr)
vote_time = time.time() - start
soft_acc = accuracy_score(y_val, vote.predict(X_val))
# same fitted members, just switch to hard voting instead of training everything again
vote.set_params(voting='hard')
hard_acc = accuracy_score(y_val, vote.predict(X_val))
vote.set_params(voting='soft')

start = time.time()
stack = StackingClassifier(make_members(SEED), final_estimator=LogisticRegression(max_iter=2000), cv=3, n_jobs=-1).fit(X_tr, y_tr)
stack_time = time.time() - start
stack_acc = accuracy_score(y_val, stack.predict(X_val))

member_preds = {name: est.predict(X_val) for name, est in vote.named_estimators_.items()}
rows = [{'model': name, 'val_acc': np.mean(pr == y_val)} for name, pr in member_preds.items()]
rows += [{'model': 'hard voting', 'val_acc': hard_acc}, {'model': 'soft voting', 'val_acc': soft_acc},
         {'model': 'stacking (LR)', 'val_acc': stack_acc}]
df5 = pd.DataFrame(rows)
df5.to_csv('results/part5_combining.csv', index=False)
print(f'voting fit time {vote_time:.1f}s, stacking fit time {stack_time:.1f}s')
print('stacking LR coef magnitude per member:', {n: np.abs(stack.final_estimator_.coef_[:, i*10:(i+1)*10]).mean().round(3)
                                                for i, n in enumerate(member_preds)})
df5.round(4)

voting fit time 737.3s, stacking fit time 1261.5s
stacking LR coef magnitude per member: {'tree': np.float64(0.455), 'rf': np.float64(0.88), 'ada': np.float64(0.019), 'hgb': np.float64(0.752)}


,model,val_acc
0,tree,0.7894
1,rf,0.8684
2,ada,0.8718
3,hgb,0.8734
4,hard voting,0.8720
5,soft voting,0.8614
6,stacking (LR),0.8712


In [46]:
# part 5 - error correlation
# how correlated are the members' errors?
names = list(member_preds)
err = np.array([member_preds[n] != y_val for n in names]).astype(float)
corr = np.corrcoef(err)
both_wrong = np.array([[np.mean(err[i].astype(bool) & err[j].astype(bool)) for j in range(len(names))] for i in range(len(names))])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, M, title in [(axes[0], corr, 'Correlation of error indicators'), (axes[1], both_wrong, 'Fraction of val points both wrong')]:
    im = ax.imshow(M, cmap='Reds', vmin=0)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=9)
    ax.set_title(title)
    ax.set_xlabel('member')
    ax.set_ylabel('member')
    plt.colorbar(im, ax=ax, fraction=0.046)
savefig('p5_error_corr')
print('oracle accuracy (at least one member right):', np.mean(err.min(0) == 0).round(4))
print('all members wrong:', np.mean(err.min(0) == 1).round(4))
print('mean error corr without the single tree:', corr[1:, 1:][np.triu_indices(3, 1)].mean().round(3))

oracle accuracy (at least one member right): 0.928
all members wrong: 0.072
mean error corr without the single tree: 0.738


In [47]:
# part 6 - final fashion runs
# test set is only used here. voting = avg of the 4 members' probs from the same seed,
# stacking = logistic regression on the members' val probs (real 3 fold stacking x5 seeds
# with adaboost inside would take hours, the real StackingClassifier is in part 5)
def fit_and_time(model, X, y):
    start = time.time()
    model.fit(X, y)
    return model, time.time() - start

def predict_ms_per_1k(model, X):
    # time prediction single threaded so its fair between models (otherwise joblib startup dominates)
    if 'n_jobs' in model.get_params():
        model.set_params(n_jobs=1)
    start = time.time()
    pred = model.predict(X)
    return pred, (time.time() - start) / len(X) * 1000 * 1000

def make_final_clf(seed):
    models = {
        'Tree (Gini)': DecisionTreeClassifier(random_state=seed, **best_gini_params),
        'Tree (entropy)': DecisionTreeClassifier(random_state=seed, **best_ent_params),
    }
    for method, nice in [('pasting', 'Pasting'), ('bagging', 'Bagging'), ('subspaces', 'Random subspaces'), ('patches', 'Random patches')]:
        fr, fc = best_frac[method]
        models[nice] = make_bagging_model(method, fr, fc, seed=seed)
    models['Random forest'] = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=seed, **best_rf_params)
    models['Extra trees'] = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=seed, **best_et_params)
    models['Gradient boosting'] = HistGradientBoostingClassifier(random_state=seed, **best_hgb_params)
    return models

hparams_clf = {
    'Tree (Gini)': str(best_gini_params), 'Tree (entropy)': str(best_ent_params),
    'Pasting': f'rows={best_frac["pasting"][0]}, 100 trees', 'Bagging': f'rows={best_frac["bagging"][0]}, 100 trees',
    'Random subspaces': f'cols={best_frac["subspaces"][1]}, 100 trees',
    'Random patches': f'rows={best_frac["patches"][0]}, cols={best_frac["patches"][1]}, 100 trees',
    'Random forest': f'{best_rf_params}, 300 trees', 'Extra trees': f'{best_et_params}, 300 trees',
    'AdaBoost': f'depth={best_ada["depth"]}, lr={best_ada["lr"]}, rounds={best_ada["rounds"]}',
    'Gradient boosting': f'lr={best_hgb_params["learning_rate"]}, max_depth={best_hgb_params["max_depth"]}, early stopping',
    'Soft voting': 'tree + RF + AdaBoost + HGB', 'Stacking (LR blend on val)': 'tree + RF + AdaBoost + HGB',
}

# adaboost is single threaded and slow, so the 5 seeds run in parallel
ada_fits = Parallel(n_jobs=N_SEEDS)(delayed(fit_and_time)(make_ada(s), X_tr, y_tr) for s in range(N_SEEDS))

rows = []
test_preds = {}
for s in range(N_SEEDS):
    fitted = {}
    for name, model in make_final_clf(s).items():
        model, t = fit_and_time(model, X_tr, y_tr)
        fitted[name] = (model, t)
    fitted['AdaBoost'] = ada_fits[s]

    for name, (model, t) in fitted.items():
        pred, ms = predict_ms_per_1k(model, X_test)
        rows.append({'method': name, 'seed': s, 'test_acc': accuracy_score(y_test, pred) * 100,
                     'macro_f1': f1_score(y_test, pred, average='macro'), 'train_time': t, 'predict_ms_per_1k': ms,
                     'nodes': count_nodes(model), 'val_acc': accuracy_score(y_val, model.predict(X_val)) * 100})
        test_preds[(name, s)] = pred

    # combos built from the members trained with this seed
    member_names = ['Tree (Gini)', 'Random forest', 'AdaBoost', 'Gradient boosting']
    combo_time = sum(fitted[n][1] for n in member_names)
    combo_nodes = sum(count_nodes(fitted[n][0]) for n in member_names)
    start = time.time()
    P_test = [fitted[n][0].predict_proba(X_test) for n in member_names]
    ms = (time.time() - start) / len(X_test) * 1e6
    P_val = [fitted[n][0].predict_proba(X_val) for n in member_names]

    pred = np.mean(P_test, axis=0).argmax(1)
    rows.append({'method': 'Soft voting', 'seed': s, 'test_acc': accuracy_score(y_test, pred) * 100,
                 'macro_f1': f1_score(y_test, pred, average='macro'), 'train_time': combo_time, 'predict_ms_per_1k': ms,
                 'nodes': combo_nodes, 'val_acc': accuracy_score(y_val, np.mean(P_val, axis=0).argmax(1)) * 100})
    test_preds[('Soft voting', s)] = pred

    start = time.time()
    blender = LogisticRegression(max_iter=2000).fit(np.hstack(P_val), y_val)
    blend_t = time.time() - start
    pred = blender.predict(np.hstack(P_test))
    rows.append({'method': 'Stacking (LR blend on val)', 'seed': s, 'test_acc': accuracy_score(y_test, pred) * 100,
                 'macro_f1': f1_score(y_test, pred, average='macro'), 'train_time': combo_time + blend_t,
                 'predict_ms_per_1k': ms, 'nodes': combo_nodes, 'val_acc': np.nan})  # val was used to train the blender
    test_preds[('Stacking (LR blend on val)', s)] = pred
    print('seed', s, 'done')

df6 = pd.DataFrame(rows)
df6.to_csv('results/part6_fashion_all_seeds.csv', index=False)

seed 0 done


seed 1 done


seed 2 done


seed 3 done


seed 4 done


In [48]:
# part 6 - fashion table
order = ['Tree (Gini)', 'Tree (entropy)', 'Pasting', 'Bagging', 'Random subspaces', 'Random patches', 'Random forest',
         'Extra trees', 'AdaBoost', 'Gradient boosting', 'Soft voting', 'Stacking (LR blend on val)']
g = df6.groupby('method')
table_clf = pd.DataFrame({
    'Method': order,
    'Best hyperparameters': [hparams_clf[m] for m in order],
    'Test accuracy (%)': [fmt(g.get_group(m).test_acc.mean(), g.get_group(m).test_acc.std()) for m in order],
    'Macro-F1': [fmt(g.get_group(m).macro_f1.mean(), g.get_group(m).macro_f1.std(), 4) for m in order],
    'Train time (s)': [fmt(g.get_group(m).train_time.mean(), g.get_group(m).train_time.std()) for m in order],
    'Predict time (ms / 1k images)': [fmt(g.get_group(m).predict_ms_per_1k.mean(), g.get_group(m).predict_ms_per_1k.std()) for m in order],
    'Model size (nodes)': [f'{g.get_group(m).nodes.mean():.0f}' for m in order],
})
table_clf.to_csv('results/table_fashion_mnist.csv', index=False)
table_clf

,Method,Best hyperparameters,Test accuracy (%),Macro-F1,Train time (s),Predict time (ms / 1k images),Model size (nodes)
0,Tree (Gini),"{'criterion': 'gini', 'ccp_alpha': 0.000618114...",77.35 ± 0.05,0.7741 ± 0.0005,3.29 ± 0.06,1.39 ± 1.54,299
1,Tree (entropy),"{'criterion': 'entropy', 'max_depth': 10}",77.21 ± 0.08,0.7711 ± 0.0007,2.37 ± 0.06,0.31 ± 0.08,759
2,Pasting,"rows=0.5, 100 trees",83.86 ± 0.07,0.8373 ± 0.0007,18.87 ± 0.94,85.15 ± 2.45,111529
3,Bagging,"rows=1.0, 100 trees",84.13 ± 0.17,0.8402 ± 0.0017,26.12 ± 1.09,84.85 ± 1.61,146518
4,Random subspaces,"cols=0.1, 100 trees",85.54 ± 0.18,0.8540 ± 0.0019,4.15 ± 0.11,29.10 ± 1.22,277312
5,Random patches,"rows=1.0, cols=0.1, 100 trees",85.54 ± 0.18,0.8540 ± 0.0019,4.16 ± 0.20,29.41 ± 1.63,277312
6,Random forest,"{'criterion': 'entropy', 'max_features': 0.1},...",85.60 ± 0.07,0.8547 ± 0.0008,7.74 ± 0.47,29.15 ± 0.45,466040
7,Extra trees,"{'criterion': 'entropy', 'max_features': None}...",86.08 ± 0.04,0.8598 ± 0.0004,17.97 ± 1.19,31.88 ± 0.72,698634
8,AdaBoost,"depth=8, lr=0.1, rounds=443",85.93 ± 0.11,0.8599 ± 0.0011,769.84 ± 1.29,109.88 ± 1.33,166206
9,Gradient boosting,"lr=0.05, max_depth=None, early stopping",86.13 ± 0.18,0.8609 ± 0.0019,40.44 ± 10.04,68.91 ± 13.13,55754


In [49]:
# part 6 - confusion matrix
# best model is picked by val accuracy, not test
val_rank = df6.groupby('method').val_acc.mean().dropna()
best_method = val_rank.idxmax()
print('best model by validation accuracy:', best_method, round(val_rank.max(), 2))
cm = confusion_matrix(y_test, test_preds[(best_method, 0)])
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
ax.set_title(f'Test confusion matrix: {best_method} (seed 0)')
savefig('p6_confusion')
per_class = cm.diagonal() / cm.sum(1)
print('per-class recall:', dict(zip(CLASS_NAMES, per_class.round(3))))
off = cm.copy()
np.fill_diagonal(off, 0)
top_pairs = np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1][:5], off.shape))[0]
for i, j in top_pairs:
    print(f'{CLASS_NAMES[i]} -> {CLASS_NAMES[j]}: {off[i, j]}')

best model by validation accuracy: Gradient boosting 87.3


per-class recall: {'T-shirt/top': np.float64(0.808), 'Trouser': np.float64(0.942), 'Pullover': np.float64(0.782), 'Dress': np.float64(0.893), 'Coat': np.float64(0.786), 'Sandal': np.float64(0.944), 'Shirt': np.float64(0.624), 'Sneaker': np.float64(0.943), 'Bag': np.float64(0.962), 'Ankle boot': np.float64(0.947)}
Shirt -> T-shirt/top: 127
Pullover -> Coat: 116
T-shirt/top -> Shirt: 115
Coat -> Pullover: 115
Shirt -> Pullover: 101


In [50]:
# part 6 - california tuning
rows = []
for method in ['pasting', 'bagging', 'subspaces', 'patches']:
    for fr in (FRACS if method != 'subspaces' else [1.0]):
        for fc in (FRACS if method in ['subspaces', 'patches'] else [1.0]):
            m = make_bagging_model(method, fr, fc, regression=True, seed=0).fit(Xc_tr, yc_tr)
            rows.append({'method': method, 'rows': fr, 'cols': fc, 'val_rmse': rmse(yc_val, m.predict(Xc_val))})
reg_bag = pd.DataFrame(rows)
best_frac_reg = {m: tuple(reg_bag[reg_bag.method == m].sort_values('val_rmse').iloc[0][['rows', 'cols']]) for m in ['pasting', 'bagging', 'subspaces', 'patches']}
print(best_frac_reg)

rows = []
REG_MF = {'1/3 p': 1 / 3, 'sqrt(p)': 'sqrt', '0.5p': 0.5, 'p': 1.0}
for label, mf in REG_MF.items():
    for name, Cls in [('rf', RandomForestRegressor), ('et', ExtraTreesRegressor)]:
        m = Cls(n_estimators=300, max_features=mf, n_jobs=-1, random_state=0).fit(Xc_tr, yc_tr)
        rows.append({'model': name, 'max_features': label, 'val_rmse': rmse(yc_val, m.predict(Xc_val))})
reg_forest = pd.DataFrame(rows)
best_rf_reg_mf = REG_MF[reg_forest[reg_forest.model == 'rf'].sort_values('val_rmse').iloc[0].max_features]
best_et_reg_mf = REG_MF[reg_forest[reg_forest.model == 'et'].sort_values('val_rmse').iloc[0].max_features]
print('rf max_features', best_rf_reg_mf, 'et max_features', best_et_reg_mf)
reg_forest.round(4)

{'pasting': (np.float64(0.7), np.float64(1.0)), 'bagging': (np.float64(1.0), np.float64(1.0)), 'subspaces': (np.float64(1.0), np.float64(0.7)), 'patches': (np.float64(0.7), np.float64(1.0))}


rf max_features 0.3333333333333333 et max_features 0.5


,model,max_features,val_rmse
0,rf,1/3 p,0.4949
1,et,1/3 p,0.4938
2,rf,sqrt(p),0.4949
3,et,sqrt(p),0.4938
4,rf,0.5p,0.4998
5,et,0.5p,0.4821
6,rf,p,0.5082
7,et,p,0.5029


In [51]:
# part 6 - california table
def make_final_reg(seed):
    models = {
        'Tree (squared_error)': DecisionTreeRegressor(random_state=seed, **best_reg_tree_params['squared_error']),
        'Tree (absolute_error)': DecisionTreeRegressor(random_state=seed, **best_reg_tree_params['absolute_error']),
    }
    for method, nice in [('pasting', 'Pasting'), ('bagging', 'Bagging'), ('subspaces', 'Random subspaces'), ('patches', 'Random patches')]:
        fr, fc = best_frac_reg[method]
        models[nice] = make_bagging_model(method, fr, fc, regression=True, seed=seed)
    models['Random forest'] = RandomForestRegressor(n_estimators=300, max_features=best_rf_reg_mf, n_jobs=-1, random_state=seed)
    models['Extra trees'] = ExtraTreesRegressor(n_estimators=300, max_features=best_et_reg_mf, n_jobs=-1, random_state=seed)
    models['AdaBoost.R2'] = AdaBoostRegressor(DecisionTreeRegressor(max_depth=best_adr_params['depth']), n_estimators=200,
                                              learning_rate=best_adr_params['lr'], random_state=seed)
    models['Gradient boosting'] = GradientBoostingRegressor(random_state=seed, **best_gbr_params)
    return models

hparams_reg = {
    'Tree (squared_error)': str(best_reg_tree_params['squared_error']), 'Tree (absolute_error)': str(best_reg_tree_params['absolute_error']),
    'Pasting': f'rows={best_frac_reg["pasting"][0]}', 'Bagging': f'rows={best_frac_reg["bagging"][0]}',
    'Random subspaces': f'cols={best_frac_reg["subspaces"][1]}',
    'Random patches': f'rows={best_frac_reg["patches"][0]}, cols={best_frac_reg["patches"][1]}',
    'Random forest': f'max_features={best_rf_reg_mf}, 300 trees', 'Extra trees': f'max_features={best_et_reg_mf}, 300 trees',
    'AdaBoost.R2': f'depth={best_adr_params["depth"]}, lr={best_adr_params["lr"]}, 200 rounds',
    'Gradient boosting': f'lr={best_gbr_params["learning_rate"]}, depth={best_gbr_params["max_depth"]}, early stopping',
    'Voting (average)': 'tree + RF + AdaBoost.R2 + GBR', 'Stacking (LR blend on val)': 'tree + RF + AdaBoost.R2 + GBR',
}

rows = []
for s in range(N_SEEDS):
    fitted = {}
    for name, model in make_final_reg(s).items():
        model, t = fit_and_time(model, Xc_tr, yc_tr)
        fitted[name] = (model, t)
        pred, ms = predict_ms_per_1k(model, Xc_test)
        rows.append({'method': name, 'seed': s, 'rmse': rmse(yc_test, pred), 'mae': mean_absolute_error(yc_test, pred),
                     'train_time': t, 'predict_ms_per_1k': ms, 'nodes': count_nodes(model)})
    member_names = ['Tree (squared_error)', 'Random forest', 'AdaBoost.R2', 'Gradient boosting']
    combo_time = sum(fitted[n][1] for n in member_names)
    combo_nodes = sum(count_nodes(fitted[n][0]) for n in member_names)
    start = time.time()
    P_test = np.column_stack([fitted[n][0].predict(Xc_test) for n in member_names])
    ms = (time.time() - start) / len(Xc_test) * 1e6
    P_val = np.column_stack([fitted[n][0].predict(Xc_val) for n in member_names])
    pred = P_test.mean(1)
    rows.append({'method': 'Voting (average)', 'seed': s, 'rmse': rmse(yc_test, pred), 'mae': mean_absolute_error(yc_test, pred),
                 'train_time': combo_time, 'predict_ms_per_1k': ms, 'nodes': combo_nodes})
    blender = LinearRegression().fit(P_val, yc_val)
    pred = blender.predict(P_test)
    rows.append({'method': 'Stacking (LR blend on val)', 'seed': s, 'rmse': rmse(yc_test, pred), 'mae': mean_absolute_error(yc_test, pred),
                 'train_time': combo_time, 'predict_ms_per_1k': ms, 'nodes': combo_nodes})
df6r = pd.DataFrame(rows)
df6r.to_csv('results/part6_california_all_seeds.csv', index=False)

order_r = ['Tree (squared_error)', 'Tree (absolute_error)', 'Pasting', 'Bagging', 'Random subspaces', 'Random patches',
           'Random forest', 'Extra trees', 'AdaBoost.R2', 'Gradient boosting', 'Voting (average)', 'Stacking (LR blend on val)']
g = df6r.groupby('method')
table_reg = pd.DataFrame({
    'Method': order_r,
    'Best hyperparameters': [hparams_reg[m] for m in order_r],
    'Test RMSE ($100k)': [fmt(g.get_group(m).rmse.mean(), g.get_group(m).rmse.std(), 4) for m in order_r],
    'Test MAE ($100k)': [fmt(g.get_group(m).mae.mean(), g.get_group(m).mae.std(), 4) for m in order_r],
    'Train time (s)': [fmt(g.get_group(m).train_time.mean(), g.get_group(m).train_time.std()) for m in order_r],
    'Predict time (ms / 1k rows)': [fmt(g.get_group(m).predict_ms_per_1k.mean(), g.get_group(m).predict_ms_per_1k.std()) for m in order_r],
    'Model size (nodes)': [f'{g.get_group(m).nodes.mean():.0f}' for m in order_r],
})
table_reg.to_csv('results/table_california.csv', index=False)
table_reg

,Method,Best hyperparameters,Test RMSE ($100k),Test MAE ($100k),Train time (s),Predict time (ms / 1k rows),Model size (nodes)
0,Tree (squared_error),"{'criterion': 'squared_error', 'max_depth': 8}",0.6538 ± 0.0005,0.4502 ± 0.0003,0.04 ± 0.00,0.13 ± 0.10,469
1,Tree (absolute_error),"{'criterion': 'absolute_error', 'max_depth': 10}",0.6597 ± 0.0011,0.4273 ± 0.0006,0.21 ± 0.00,0.10 ± 0.01,1415
2,Pasting,rows=0.7,0.5096 ± 0.0013,0.3313 ± 0.0005,0.80 ± 0.01,18.84 ± 0.35,1666574
3,Bagging,rows=1.0,0.5102 ± 0.0014,0.3323 ± 0.0006,0.69 ± 0.01,18.12 ± 0.23,1508868
4,Random subspaces,cols=0.7,0.5148 ± 0.0068,0.3551 ± 0.0057,0.83 ± 0.02,22.09 ± 0.41,2394492
5,Random patches,"rows=0.7, cols=1.0",0.5096 ± 0.0013,0.3313 ± 0.0005,0.80 ± 0.04,18.99 ± 0.47,1666574
6,Random forest,"max_features=0.3333333333333333, 300 trees",0.5024 ± 0.0010,0.3343 ± 0.0009,0.66 ± 0.02,53.80 ± 3.22,4571037
7,Extra trees,"max_features=0.5, 300 trees",0.4929 ± 0.0020,0.3224 ± 0.0014,0.58 ± 0.05,70.37 ± 3.98,7263776
8,AdaBoost.R2,"depth=10, lr=1.0, 200 rounds",0.5032 ± 0.0012,0.3661 ± 0.0009,6.40 ± 0.04,21.35 ± 0.23,217944
9,Gradient boosting,"lr=0.03, depth=7, early stopping",0.4696 ± 0.0053,0.3088 ± 0.0042,21.73 ± 7.57,13.27 ± 3.56,112114


In [52]:
# bonus - cart from scratch
class MyCART:
    def __init__(self, criterion='gini', max_depth=None, min_samples_leaf=1):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf

    def _impurity(self, y):
        p = np.bincount(y, minlength=self.n_classes) / len(y)
        if self.criterion == 'gini':
            return 1 - np.sum(p ** 2)
        p = p[p > 0]
        return -np.sum(p * np.log2(p))

    def _best_split(self, X, y):
        best_feat, best_thr, best_gain = None, None, 0.0
        parent = self._impurity(y)
        n = len(y)
        for j in range(X.shape[1]):
            vals = np.unique(X[:, j])
            for k in range(len(vals) - 1):
                thr = (vals[k] + vals[k + 1]) / 2
                left = X[:, j] <= thr
                nl = left.sum()
                nr = n - nl
                if nl < self.min_samples_leaf or nr < self.min_samples_leaf:
                    continue
                gain = parent - nl / n * self._impurity(y[left]) - nr / n * self._impurity(y[~left])
                if gain > best_gain + 1e-12:
                    best_feat, best_thr, best_gain = j, thr, gain
        return best_feat, best_thr

    def _build(self, X, y, depth):
        node = {'probs': np.bincount(y, minlength=self.n_classes) / len(y)}
        # stop if pure, too deep, or too small to split
        if len(np.unique(y)) == 1 or (self.max_depth is not None and depth >= self.max_depth) or len(y) < 2 * self.min_samples_leaf:
            return node
        f, thr = self._best_split(X, y)
        if f is None:
            return node
        left = X[:, f] <= thr
        node['feature'], node['threshold'] = f, thr
        node['left'] = self._build(X[left], y[left], depth + 1)
        node['right'] = self._build(X[~left], y[~left], depth + 1)
        return node

    def fit(self, X, y):
        self.n_classes = int(y.max()) + 1
        self.root = self._build(X, y, 0)
        return self

    def _predict_one(self, x):
        node = self.root
        while 'feature' in node:
            node = node['left'] if x[node['feature']] <= node['threshold'] else node['right']
        return node['probs']

    def predict(self, X):
        return np.array([self._predict_one(x).argmax() for x in X])

    def n_leaves(self, node=None):
        node = self.root if node is None else node
        if 'feature' not in node:
            return 1
        return self.n_leaves(node['left']) + self.n_leaves(node['right'])

Xm, ym = make_moons(n_samples=600, noise=0.3, random_state=SEED)
Xm = Xm.astype(np.float32)  # sklearn trees work in float32, so do the same to get the same thresholds
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.3, random_state=SEED)

rows = []
for crit in ['gini', 'entropy']:
    for d in [1, 3, 6, None]:
        for msl in [1, 10]:
            mine = MyCART(crit, d, msl).fit(Xm_tr, ym_tr)
            sk = DecisionTreeClassifier(criterion=crit, max_depth=d, min_samples_leaf=msl, random_state=SEED).fit(Xm_tr, ym_tr)
            pm, ps = mine.predict(Xm_te), sk.predict(Xm_te)
            rows.append({'criterion': crit, 'max_depth': str(d), 'min_samples_leaf': msl,
                         'my test acc': np.mean(pm == ym_te), 'sklearn test acc': np.mean(ps == ym_te),
                         'prediction agreement': np.mean(pm == ps), 'my leaves': mine.n_leaves(), 'sklearn leaves': sk.get_n_leaves()})
df_bonus = pd.DataFrame(rows)
df_bonus.to_csv('results/bonus_cart_check.csv', index=False)
df_bonus.round(4)

,criterion,max_depth,min_samples_leaf,my test acc,sklearn test acc,prediction agreement,my leaves,sklearn leaves
0,gini,1,1,0.8000,0.8000,1.0000,2,2
1,gini,1,10,0.8000,0.8000,1.0000,2,2
2,gini,3,1,0.8889,0.8889,1.0000,6,6
3,gini,3,10,0.8889,0.8889,1.0000,6,6
4,gini,6,1,0.9000,0.9056,0.9944,19,19
5,gini,6,10,0.8944,0.8944,1.0000,17,17
6,gini,None,1,0.8833,0.8778,0.9500,49,49
7,gini,None,10,0.8944,0.8944,1.0000,22,22
8,entropy,1,1,0.8000,0.8000,1.0000,2,2
9,entropy,1,10,0.8000,0.8000,1.0000,2,2


In [53]:
# bonus - plot
xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (name, model) in zip(axes, [('my CART', MyCART('gini', 4).fit(Xm_tr, ym_tr)),
                                    ('sklearn', DecisionTreeClassifier(max_depth=4, random_state=SEED).fit(Xm_tr, ym_tr))]):
    ax.contourf(xx, yy, model.predict(grid).reshape(xx.shape), alpha=0.3, cmap='coolwarm')
    ax.scatter(Xm_tr[:, 0], Xm_tr[:, 1], c=ym_tr, cmap='coolwarm', s=10, edgecolor='k', lw=0.3)
    ax.set_title(f'{name}: Gini, max_depth=4')
    ax.set_xlabel('feature 1')
    ax.set_ylabel('feature 2')
savefig('bonus_boundaries')